### 0. 環境建置

此 notebook 以 Google Colab 為主要執行環境。第一次執行時會安裝所需套件並**自動重啟一次 Python runtime**；重啟後請再按一次 **Run all**。這可避免 Colab 在安裝套件後出現 NumPy / compiled package ABI 不一致（`numpy.dtype size changed`）的問題。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

setup_marker = Path('/content/.rag2_environment_ready')

if not setup_marker.exists():
    packages = [
        'langchain-community',
        'faiss-cpu',
        'sentence-transformers',
        'huggingface_hub',
        'gradio',
        'aisuite',
        'groq',
    ]
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--upgrade-strategy', 'only-if-needed',
        *packages,
    ])
    setup_marker.touch()
    print('✅ 套件安裝完成。現在自動重啟 runtime；重啟後請再按一次 Run all。')
    os.kill(os.getpid(), 9)
else:
    print('✅ 環境已安裝，可繼續執行。')

### 1. 讀入你打造好的 vector dataset

原作業提交時使用的 Google Drive 共用連結已失效。請先執行 **RAG (1)** 產生 `faiss_db.zip`，再於此 notebook 手動上傳該檔案。

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_candidates:
    raise FileNotFoundError('Please upload the faiss_db.zip generated by RAG (1).')
zip_name = zip_candidates[0]
print(f'Using vector database archive: {zip_name}')

In [ ]:
import zipfile
with zipfile.ZipFile(zip_name, 'r') as archive:
    archive.extractall('.')
print('Vector database extracted.')

### 2. 引入必要套件

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import gradio as gr

### 3. 自訂 EmbeddingGemma embedding 類別

In [ ]:
import os
from google.colab import userdata
hf_token = userdata.get('HuggingFace')

In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
class EmbeddingGemmaEmbeddings(HuggingFaceEmbeddings):
    def __init__(self, **kwargs):
        super().__init__(
            model_name='google/embeddinggemma-300m',
            encode_kwargs={'normalize_embeddings': True},
            **kwargs
        )

    def embed_documents(self, texts):
        texts = [f'title: RPG | text: {t}' for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        return super().embed_query(f'task: search result | query: {text}')

### 4. 載入 `faiss_db`

In [ ]:
embedding_model = EmbeddingGemmaEmbeddings()
vectorstore = FAISS.load_local(
    'faiss_db',
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

### 5. 設定 LLM

In [ ]:
import aisuite as ai
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY'] = api_key
model = 'groq:openai/gpt-oss-120b'
client = ai.Client()

### 6. `prompt` 設計

In [ ]:
system_prompt = '你是一款 RPG 遊戲的提示員，請根據遊戲資料來回應玩家的問題。請親切、簡潔並附帶具體建議。請用台灣習慣的中文回應。'

prompt_template = '''
根據下列遊戲資料：
{retrieved_chunks}

請嚴格遵守以下規則，回答玩家的問題：{question}

1. **只回答遊戲相關問題**：你的唯一職責是擔任這款 RPG 遊戲的提示員。
2. **處理無關問題**：如果玩家的問題與這款遊戲完全無關，不回答實質內容，回覆：「這個問題好像跟我們的冒險沒有關係耶，我只知道這個遊戲世界裡的秘密喔！」
3. **處理資料不足的問題**：如果問題與遊戲相關，但檢索資料不足，回覆：「這部分的線索我目前沒有，你可能需要自行在遊戲世界中探索更多細節」。
4. **回答資料充足的問題**：如果資料充足，請根據資料直接回答。
'''

### 7. 使用 RAG 來回應

In [ ]:
def chat_with_rag(user_input):
    docs = retriever.invoke(user_input)
    retrieved_chunks = '\n\n'.join(doc.page_content for doc in docs)
    final_prompt = prompt_template.format(
        retrieved_chunks=retrieved_chunks,
        question=user_input,
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': final_prompt},
        ],
    )
    return response.choices[0].message.content

### 8. 用 Gradio 打造 Web App

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown('# 🪽 RPG輔助小天使')
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder='請輸入你的問題...')

    def respond(message, history):
        response = chat_with_rag(message)
        history = history or []
        history.extend([
            {'role': 'user', 'content': message},
            {'role': 'assistant', 'content': response},
        ])
        return '', history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch(debug=True)